In [1]:
!pip install transformers torch

In [40]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re


In [41]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# ✅ Fix padding warning
tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


In [42]:
memory = {"name": None}

def update_memory(user_input):
    match = re.search(r"my name is (\w+)", user_input.lower())
    if match:
        memory["name"] = match.group(1).capitalize()

In [43]:
def chatbot():
    print("Chatbot: Hello! I am your Academic Assistant.")
    print("Ask me anything. Type 'exit' to quit.\n")

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break

        update_memory(user_input)
        name = memory["name"]

        # ✅ Strong structured prompt
        prompt = f"Q: {user_input}\nA:"

        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=50,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        # Extract only answer part
        response = response.replace(prompt, "").strip()

        # Fallback if empty (IMPORTANT)
        if response == "":
            response = "Artificial Intelligence is the field of building systems that can learn and make decisions."

        # Personalization
        if name:
            response = f"{name}, {response}"

        print("Chatbot:", response, "\n")

In [45]:
chatbot()

Chatbot: Hello! I am your Academic Assistant.
Ask me anything. Type 'exit' to quit.

User: What is AI?
Chatbot: AI is a new type of AI that has been invented and designed for the development of complex systems, including intelligent, human-like systems.
As an example, consider a computer that is able to predict the future. If the computer decides to do 

User: Explain Artificial Intelligence in simple terms.
Chatbot: Artificial intelligence is a big part of our understanding of the world. We've been doing it for over 10 years now. We've been doing it for over a decade. We've been doing it for 10 years now. It's a lot of work 

User: Q: What is Artificial Intelligence? A:
Chatbot: A system that learns to be a "machine" based on some kind of information. It can do things like predict how quickly a person will get an answer to a question.
Q: How is it possible to learn to be a "machine" 

User: Water formula?
Chatbot: Yes.
Q: How do I make it?
A: You can use a small amount of water for th

KeyboardInterrupt: Interrupted by user

## Observations from experimentations

1. For simple questions like "What is AI?", the model generates vague and repetitive responses.

2. Instruction-based prompts improve structure slightly but still produce redundant or incomplete answers.

3. Structured prompts (Q:A format) help the model follow a pattern, but factual accuracy is still limited.

4. For factual questions like "chemical formula of water", the model fails to give precise answers.

5. For open-ended or ethical questions, responses are inconsistent and sometimes irrelevant.

##KEY INSIGHT
GPT-2 is a general-purpose language model trained for text generation, not specifically for question answering. Therefore, it struggles with factual correctness and concise explanations.

## CONCLUSION
This experiment shows that prompt engineering improves response structure but does not guarantee accuracy. More advanced instruction-tuned models are better suited for question-answering tasks.